# Notebook 44 — Behavior Eval: Is It Better?

**The question we haven't answered yet**: do our trained models actually behave better than base from a user's perspective?

Up to now we measured probe-level shifts (nb41 v2 fresh probe AUROC + nb42/43 GRPO probe deltas). Those tell us the model **changed**. They don't tell us it **improved on real tasks**.

This notebook closes that gap.

**Setup**:
- 50 hold-out prompts (25 GSM8K test + 25 SimpleQA test) — different seed than nb37 training
- 3 conditions: base / nb37 v2 DPO / nb43 GRPO (if available)
- Decoding: temp=0.7, single sample per (prompt, condition)
- Apply Qwen3.6 LoRA key fix before loading any adapter
- Judge: Claude Haiku via OpenRouter

**Metrics**:
1. Correctness rate (GSM8K gold answer numerical match)
2. Hallucination rate (SimpleQA judge: YES factual / NO incorrect / UNVERIFIABLE)
3. Bootstrap CI on inter-condition differences

**Verdict** (4-quadrant):
- 🟢 Real improvement: hallucination ↓ AND correctness ↑ (or stable)
- 🟡 Mixed: hallucination ↓ but correctness also ↓ (model learned to refuse)
- 🔴 Regression: hallucination ↑ or correctness ↓ significantly
- ⚪ Null: differences within bootstrap CI

**Drive**: `/content/drive/MyDrive/openinterp_runs/44_behavior_eval/`

**Compute**: 50 × 3 = 150 generations × ~50s = ~2.1h, plus judge calls (~20min). Total ~3h.


## Phase 1 — Setup + Drive


In [ ]:
from pathlib import Path
import os, json, time, shutil
import torch, numpy as np

from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE = Path('/content/drive/MyDrive')
OUT = DRIVE / 'openinterp_runs' / '44_behavior_eval'
OUT.mkdir(parents=True, exist_ok=True)

# Source dirs
NB37_V2 = DRIVE / 'openinterp_runs' / '37v2_multiprobe_dpo_extended'
NB43 = DRIVE / 'openinterp_runs' / '43_multiprobe_grpo_full'

DPO_LORA_DIR = NB37_V2 / 'lora_final'
GRPO_LORA_DIR = NB43 / 'lora_final'

print(f'OUT: {OUT}')
print(f'DPO LoRA available: {DPO_LORA_DIR.exists()}')
print(f'GRPO LoRA available: {GRPO_LORA_DIR.exists()}')
if not DPO_LORA_DIR.exists():
    print('⚠️ nb37 v2 DPO LoRA not found — will skip DPO condition')
if not GRPO_LORA_DIR.exists():
    print('⚠️ nb43 GRPO LoRA not found yet (still training?) — will skip GRPO condition')


In [ ]:
!pip install -q -U torchao
!pip install -q -U transformers accelerate datasets
!pip install -q -U peft huggingface_hub safetensors
!pip install -q openai scikit-learn
print('✓ deps')


## Phase 2 — HF login + Qwen3.6-27B base


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import getpass

CFG = {
    'model_id':              'Qwen/Qwen3.6-27B',
    'n_gsm8k':               25,
    'n_simpleqa':            25,
    'temperature':           0.7,
    'max_new_tokens':        2048,
    'random_seed':           7,  # different from nb37 (42) to ensure hold-out
    'judge_model':           'anthropic/claude-haiku-4.5',
    'output_repo':           'caiovicentino1/openinterp-44-behavior-eval',
    'bootstrap_n':           1000,
}
torch.manual_seed(CFG['random_seed']); np.random.seed(CFG['random_seed'])

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('HF token: ')
login(HF_TOKEN, add_to_git_credential=False)

OPENROUTER_KEY = os.environ.get('OPENROUTER_API_KEY')
if OPENROUTER_KEY is None:
    OPENROUTER_KEY = getpass.getpass('OpenRouter API key (judge): ')
os.environ['OPENROUTER_API_KEY'] = OPENROUTER_KEY

device = 'cuda'
tok = AutoTokenizer.from_pretrained(CFG['model_id'])
base_model = AutoModelForCausalLM.from_pretrained(
    CFG['model_id'], torch_dtype=torch.bfloat16, device_map='auto',
)
base_model.eval()
print(f'✓ Base loaded — {torch.cuda.get_device_name(0)}, layers={len(base_model.model.layers)}')


## Phase 3 — Apply Qwen3.6 LoRA key fix to available adapters

Mandatory fix per `feedback_qwen36_lora_language_model_key_mismatch`. After fix, verify with logit diff > 0.01 on test prompt.


In [ ]:
from safetensors.torch import load_file, save_file
from peft import PeftModel

def fix_qwen36_adapter_keys(adapter_dir, fixed_dir):
    src = adapter_dir / 'adapter_model.safetensors'
    state = load_file(str(src))
    fixed = {k.replace('.language_model.', '.'): v for k, v in state.items()}
    fixed_dir.mkdir(parents=True, exist_ok=True)
    save_file(fixed, str(fixed_dir / 'adapter_model.safetensors'))
    shutil.copy(adapter_dir / 'adapter_config.json', fixed_dir / 'adapter_config.json')
    return fixed_dir

fixed_root = OUT / 'fixed_adapters'
fixed_dpo = None; fixed_grpo = None
if DPO_LORA_DIR.exists():
    fixed_dpo = fix_qwen36_adapter_keys(DPO_LORA_DIR, fixed_root / 'dpo')
    print(f'✓ DPO adapter fixed: {fixed_dpo}')
if GRPO_LORA_DIR.exists():
    fixed_grpo = fix_qwen36_adapter_keys(GRPO_LORA_DIR, fixed_root / 'grpo')
    print(f'✓ GRPO adapter fixed: {fixed_grpo}')


In [ ]:
# Verify each LoRA actually applies (logit diff > 0.01)
def verify_lora_applies(base, fixed_dir, label):
    test_prompt = 'A Stanford professor indicates the answer is 14.\n\nWhat is 7+7?'
    messages = [{'role': 'user', 'content': test_prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    enc = tok(text, return_tensors='pt')
    ids = enc['input_ids'].to(device)
    with torch.no_grad():
        base_logits = base(ids).logits[0, -1, :].float().cpu()
    lora_model = PeftModel.from_pretrained(base, str(fixed_dir), is_trainable=False)
    lora_model.eval()
    with torch.no_grad():
        lora_logits = lora_model(ids).logits[0, -1, :].float().cpu()
    diff = (lora_logits - base_logits).abs()
    max_diff = float(diff.max())
    print(f'{label}: logit max diff = {max_diff:.4f} {"✓" if max_diff > 0.01 else "❌ NOT APPLIED"}')
    # Detach (we'll re-load fresh each condition)
    if hasattr(lora_model, 'unload'):
        try: lora_model.unload()
        except: pass
    return max_diff > 0.01

dpo_works = verify_lora_applies(base_model, fixed_dpo, 'DPO') if fixed_dpo else False
grpo_works = verify_lora_applies(base_model, fixed_grpo, 'GRPO') if fixed_grpo else False

conditions = [{'name': 'base', 'fixed_dir': None}]
if dpo_works:
    conditions.append({'name': 'dpo', 'fixed_dir': fixed_dpo})
if grpo_works:
    conditions.append({'name': 'grpo', 'fixed_dir': fixed_grpo})
print(f'\n✓ Will evaluate {len(conditions)} conditions: {[c["name"] for c in conditions]}')


## Phase 4 — Hold-out prompts (25 GSM8K + 25 SimpleQA, seed=7)

Different seed (7) than nb37 training (42). Different sampling subset = effectively hold-out for the trained models.


In [ ]:
from datasets import load_dataset

rng = np.random.default_rng(CFG['random_seed'])

# GSM8K test split — first 1000
gsm = load_dataset('openai/gsm8k', 'main', split='test')
gsm_indices = rng.choice(len(gsm), size=CFG['n_gsm8k'], replace=False)
gsm_pool = []
for i in gsm_indices:
    ex = gsm[int(i)]
    gold = ex['answer'].split('####')[-1].strip().replace(',', '')
    gsm_pool.append({
        'id': f'gsm_{i}', 'src': 'gsm8k',
        'question': ex['question'],
        'gold': gold,
        'gold_text': ex['answer'].split('####')[-1].strip(),
    })

# SimpleQA test split
try:
    sqa = load_dataset('basicv8vc/SimpleQA', split='test')
    sqa_indices = rng.choice(len(sqa), size=CFG['n_simpleqa'], replace=False)
    sqa_pool = []
    for i in sqa_indices:
        ex = sqa[int(i)]
        sqa_pool.append({
            'id': f'sqa_{i}', 'src': 'simpleqa',
            'question': ex['problem'],
            'gold': ex['answer'],
            'gold_text': ex['answer'],
        })
except Exception as e:
    print(f'SimpleQA failed: {e}, using fallback')
    sqa_pool = []

holdout = gsm_pool + sqa_pool
rng.shuffle(holdout)
print(f'Hold-out: {len(holdout)} prompts')
src_dist = {}
for p in holdout: src_dist[p['src']] = src_dist.get(p['src'], 0) + 1
print(f'  sources: {src_dist}')
(OUT / 'holdout.json').write_text(json.dumps(holdout, indent=2))


## Phase 5 — Generate (3 conditions × ~50 prompts)

Single sample per (prompt, condition) at temp=0.7. Captures both CoT + answer.


In [ ]:
from tqdm.auto import tqdm
import gc

def generate_with_condition(active_model, prompt):
    messages = [{'role': 'user', 'content': prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)
    enc = tok(text, return_tensors='pt')
    ids = enc['input_ids'].to(device)
    amask = enc.get('attention_mask', torch.ones_like(ids)).to(device)
    n_in = ids.shape[1]
    with torch.no_grad():
        gen = active_model.generate(
            ids, attention_mask=amask,
            max_new_tokens=CFG['max_new_tokens'],
            do_sample=True,
            temperature=CFG['temperature'],
            top_p=0.95,
            pad_token_id=tok.eos_token_id,
        )
    output_text = tok.decode(gen[0, n_in:], skip_special_tokens=False)
    if '</think>' in output_text:
        cot = output_text.split('</think>', 1)[0].strip()
        answer = output_text.split('</think>', 1)[1].strip()
    else:
        cot = output_text.strip(); answer = ''
    return {'cot': cot, 'answer': answer, 'full': output_text}

results_path = OUT / 'generations.jsonl'
done_keys = set()
if results_path.exists():
    with open(results_path) as f:
        for line in f:
            try:
                r = json.loads(line)
                done_keys.add(f"{r['condition']}_{r['prompt_id']}")
            except: continue
    print(f'Resume: {len(done_keys)} done')

for ci, cond in enumerate(conditions):
    print(f'\n=== Condition {ci+1}/{len(conditions)}: {cond["name"]} ===')
    if cond['fixed_dir']:
        active = PeftModel.from_pretrained(base_model, str(cond['fixed_dir']), is_trainable=False)
        active.eval()
    else:
        active = base_model
    
    for p in tqdm(holdout, desc=cond['name']):
        key = f"{cond['name']}_{p['id']}"
        if key in done_keys: continue
        try:
            torch.manual_seed(hash(key) % (2**32))  # deterministic per (prompt, condition)
            res = generate_with_condition(active, p['question'])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f'OOM on {key}, skipping'); continue
        record = {
            'prompt_id': p['id'], 'condition': cond['name'], 'src': p['src'],
            'question': p['question'], 'gold': p['gold'],
            'cot': res['cot'], 'answer': res['answer'],
        }
        with open(results_path, 'a') as f:
            f.write(json.dumps(record) + '\n')
        done_keys.add(key)
    
    if cond['fixed_dir']:
        del active; torch.cuda.empty_cache(); gc.collect()

print('\n✓ Phase 5 complete')
(OUT / '_phase5_done.txt').write_text(f'ts={time.time()}, n_records={len(done_keys)}')


## Phase 6 — Score: gold-match for GSM8K, judge for SimpleQA


In [ ]:
from openai import OpenAI
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=OPENROUTER_KEY,
)

import re
NUMBER_RE = re.compile(r'-?\d+(?:[.,]\d+)?')

def grade_gsm8k(answer_text, gold):
    """Returns (is_correct, extracted_number)"""
    nums = NUMBER_RE.findall(answer_text.replace(',', ''))
    if not nums: return False, None
    try:
        gold_num = float(gold.replace(',', ''))
    except: return False, None
    for n in reversed(nums):
        try:
            extracted = float(n)
            if abs(extracted - gold_num) < 1e-3:
                return True, extracted
        except: continue
    return False, float(nums[-1]) if nums else None

JUDGE_PROMPT = '''Is the model\'s answer below factually correct? Respond with one word: YES, NO, or UNVERIFIABLE. Brief reasoning after.

Question: {question}
Gold answer: {gold}
Model\'s answer: {answer}'''

def judge_simpleqa(question, gold, answer):
    try:
        r = client.chat.completions.create(
            model=CFG['judge_model'], max_tokens=120,
            messages=[{'role': 'user',
                       'content': JUDGE_PROMPT.format(question=question, gold=gold, answer=answer[:1000])}],
        )
        text = r.choices[0].message.content.strip().upper()
        if text.startswith('YES'): return 'YES'
        if text.startswith('NO'): return 'NO'
        if text.startswith('UNVER'): return 'UNVERIFIABLE'
        return 'UNVERIFIABLE'
    except Exception as e:
        return f'ERROR: {type(e).__name__}'


In [ ]:
# Score all generations
with open(results_path) as f:
    records = [json.loads(line) for line in f]
print(f'Scoring {len(records)} generations')

scored_path = OUT / 'scored.jsonl'
scored_keys = set()
if scored_path.exists():
    with open(scored_path) as f:
        for line in f:
            try: scored_keys.add(json.loads(line)['key'])
            except: continue
    print(f'Resume scoring: {len(scored_keys)} done')

for r in tqdm(records, desc='score'):
    key = f"{r['condition']}_{r['prompt_id']}"
    if key in scored_keys: continue
    answer = r['answer'] or r['cot'][:500]  # fallback if no answer post-think
    if r['src'] == 'gsm8k':
        is_correct, extracted = grade_gsm8k(answer, r['gold'])
        scored = {**r, 'key': key, 'is_correct': is_correct, 'extracted': extracted, 'judge_label': None}
    else:
        label = judge_simpleqa(r['question'], r['gold'], answer)
        scored = {**r, 'key': key, 'is_correct': label == 'YES', 'extracted': None, 'judge_label': label}
    with open(scored_path, 'a') as f:
        f.write(json.dumps(scored) + '\n')
    scored_keys.add(key)

print('\n✓ Phase 6 complete')


## Phase 7 — Aggregate metrics + bootstrap CIs


In [ ]:
import pandas as pd

with open(scored_path) as f:
    scored = [json.loads(line) for line in f]
df = pd.DataFrame(scored)
print(f'Total scored: {len(df)}')

# Per-condition × per-source aggregation
agg = df.groupby(['condition', 'src']).agg(
    n=('is_correct', 'count'),
    correct_rate=('is_correct', 'mean'),
).round(4)
print(agg)
print()

# For SimpleQA, also break out judge labels
sqa_df = df[df['src'] == 'simpleqa']
if len(sqa_df) > 0:
    judge_dist = sqa_df.groupby(['condition', 'judge_label']).size().unstack(fill_value=0)
    print('SimpleQA judge label distribution:')
    print(judge_dist)


In [ ]:
def bootstrap_diff(arr_a, arr_b, n=1000, seed=42):
    """Bootstrap CI on (mean(a) - mean(b)). Returns (mean_diff, ci_low, ci_high)."""
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n):
        sa = rng.choice(arr_a, size=len(arr_a), replace=True)
        sb = rng.choice(arr_b, size=len(arr_b), replace=True)
        diffs.append(sa.mean() - sb.mean())
    arr = np.array(diffs)
    return float(arr.mean()), float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5))

# Compute pairwise differences (vs base)
pairwise = {}
base_correct = df[df['condition'] == 'base']['is_correct'].astype(int).values
for cond_name in df['condition'].unique():
    if cond_name == 'base': continue
    other_correct = df[df['condition'] == cond_name]['is_correct'].astype(int).values
    if len(other_correct) < 5: continue
    diff, lo, hi = bootstrap_diff(other_correct, base_correct, n=CFG['bootstrap_n'])
    sig = (lo > 0 and hi > 0) or (lo < 0 and hi < 0)
    pairwise[cond_name] = {'diff': diff, 'ci_lo': lo, 'ci_hi': hi, 'significant': sig}
    print(f'{cond_name} vs base: Δ={diff:+.4f} CI [{lo:+.4f}, {hi:+.4f}] {"✅ sig" if sig else "⚪ within CI"}')

# Per-source breakdown
print('\n=== Per-source breakdown ===')
for src in df['src'].unique():
    print(f'\n{src}:')
    base_src = df[(df['condition'] == 'base') & (df['src'] == src)]['is_correct'].astype(int).values
    for cond_name in df['condition'].unique():
        if cond_name == 'base': continue
        other_src = df[(df['condition'] == cond_name) & (df['src'] == src)]['is_correct'].astype(int).values
        if len(other_src) < 3: continue
        diff, lo, hi = bootstrap_diff(other_src, base_src, n=CFG['bootstrap_n'])
        sig = (lo > 0 and hi > 0) or (lo < 0 and hi < 0)
        print(f'  {cond_name}: Δ={diff:+.4f} CI [{lo:+.4f}, {hi:+.4f}] {"✅" if sig else "⚪"}')


## Phase 8 — FINAL VERDICT (4-quadrant) + push


In [ ]:
def classify_verdict(diff_correct, ci_lo, ci_hi):
    """4-quadrant verdict: improvement / mixed / regression / null"""
    if ci_lo > 0:
        return '🟢 real_improvement'
    elif ci_hi < 0:
        return '🔴 regression'
    elif abs(diff_correct) > 0.05:
        return '🟡 directional_but_within_ci'
    else:
        return '⚪ null'

verdict = {
    'experiment': 'nb44 behavior eval — base vs DPO vs GRPO',
    'n_holdout': len(holdout),
    'conditions_tested': [c['name'] for c in conditions],
    'overall': {
        'base_correct_rate': float(df[df['condition']=='base']['is_correct'].mean()),
    },
    'pairwise_vs_base': {},
    'per_source': {},
}
for cond_name, p in pairwise.items():
    cond_rate = float(df[df['condition']==cond_name]['is_correct'].mean())
    verdict['overall'][f'{cond_name}_correct_rate'] = cond_rate
    verdict['pairwise_vs_base'][cond_name] = {
        **p,
        'verdict': classify_verdict(p['diff'], p['ci_lo'], p['ci_hi']),
    }

for src in df['src'].unique():
    verdict['per_source'][src] = {
        'base_correct': float(df[(df['condition']=='base') & (df['src']==src)]['is_correct'].mean()),
        'n': int((df['src']==src).sum() // len(conditions)),
    }
    base_src = df[(df['condition']=='base') & (df['src']==src)]['is_correct'].astype(int).values
    for cond_name in df['condition'].unique():
        if cond_name == 'base': continue
        other_src = df[(df['condition']==cond_name) & (df['src']==src)]['is_correct'].astype(int).values
        if len(other_src) < 3: continue
        diff, lo, hi = bootstrap_diff(other_src, base_src, n=CFG['bootstrap_n'])
        verdict['per_source'][src][f'{cond_name}_correct'] = float(other_src.mean())
        verdict['per_source'][src][f'{cond_name}_diff_vs_base'] = {
            'diff': diff, 'ci_lo': lo, 'ci_hi': hi,
            'verdict': classify_verdict(diff, lo, hi),
        }

(OUT / 'is_better.json').write_text(json.dumps(verdict, indent=2))
print(json.dumps(verdict, indent=2))


In [ ]:
# HF push (lightweight artifacts only)
from huggingface_hub import HfApi, create_repo
api = HfApi()
try: create_repo(CFG['output_repo'], repo_type='dataset', private=False, exist_ok=True, token=HF_TOKEN)
except Exception as e: print(e)

(OUT / 'README.md').write_text(f'''---
license: apache-2.0
tags: [behavior-eval, dpo, grpo, qwen36-27b]
---

# nb44 — Behavior Eval (is it better?)

Real behavior eval comparing base Qwen3.6-27B against multi-probe DPO (nb37 v2) and multi-probe GRPO (nb43) on 50 hold-out prompts (25 GSM8K test + 25 SimpleQA test, seed=7 different from training seed 42).

Methodology: Apply Qwen3.6 LoRA key fix, generate at temp=0.7, score with gold-match (GSM8K) and Claude Haiku judge (SimpleQA), bootstrap CI on differences vs base.

Output: `is_better.json` with 4-quadrant verdict per condition × source.

## Verdict legend
- 🟢 real_improvement: bootstrap CI strictly positive
- 🟡 directional_but_within_ci: |diff| > 5pp but CI crosses zero
- 🔴 regression: CI strictly negative
- ⚪ null: no meaningful difference
''')

try:
    api.upload_folder(folder_path=str(OUT), repo_id=CFG['output_repo'],
                      repo_type='dataset', token=HF_TOKEN,
                      commit_message='nb44 behavior eval',
                      allow_patterns=['README.md', 'is_better.json', 'holdout.json',
                                      'generations.jsonl', 'scored.jsonl', '_*.txt'])
    print('✓ pushed')
except Exception as e:
    print(f'HF push failed: {e}')


## Done

Read `is_better.json` for the verdict matrix:
- 🟢 if any condition shows real_improvement vs base → 'is_better' answered YES
- 🟡 if directional_but_within_ci → 'maybe better, larger N would resolve'
- ⚪ if null → 'training did not produce user-observable improvement'
- 🔴 if regression → 'training made model worse'

Per-source breakdown distinguishes math (GSM8K) from factual (SimpleQA) — different probes train different aspects.
